# 📊 Credit Risk — Exploratory Data Analysis

Full EDA of the Loan Default dataset before modelling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('../data/Loan_default.csv')
print(df.shape)
df.head()

## 1 · Basic Info & Missing Values

In [ ]:
df.info()
print('\nMissing values per column:')
print(df.isnull().sum())
df.describe()

## 2 · Target Distribution

In [ ]:
counts = df['Default'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie(counts, labels=['No Default','Default'],
            autopct='%1.1f%%', colors=['#00c6ff','#ff4b4b'], startangle=90)
axes[0].set_title('Class Distribution')

sns.countplot(x='Default', data=df, palette=['#00c6ff','#ff4b4b'], ax=axes[1])
axes[1].set_xticklabels(['No Default','Default'])
axes[1].set_title('Default Count')

plt.tight_layout()
plt.show()
print(f'Imbalance ratio  →  {counts[0]/counts[1]:.2f}:1')

## 3 · Numerical Feature Distributions by Default Status

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.drop('Default').tolist()
n_cols = 3
n_rows = -(-len(num_cols) // n_cols)  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, hue='Default',
                 palette={0:'#00c6ff', 1:'#ff4b4b'}, kde=True, bins=30, ax=axes[i])
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Features vs Default', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4 · Categorical Features vs Default

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(len(cat_cols), 1, figsize=(10, len(cat_cols) * 4))
if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, hue='Default',
                  palette={0:'#00c6ff', 1:'#ff4b4b'}, order=order, ax=ax)
    ax.set_title(f'{col} vs Default')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 5 · Default Rate by Category

In [ ]:
for col in cat_cols:
    rates = df.groupby(col)['Default'].mean().sort_values(ascending=False)
    print(f'\n--- Default Rate by {col} ---')
    print((rates * 100).round(2).astype(str) + '%')

## 6 · Correlation Heatmap

In [ ]:
corr = df[num_cols + ['Default']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 7 · Boxplots — Key Features vs Default

In [ ]:
key_features = [f for f in ['Age','Income','LoanAmount','CreditScore','DTIRatio','InterestRate'] if f in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes.flatten(), key_features):
    sns.boxplot(data=df, x='Default', y=col,
                palette={0:'#00c6ff', 1:'#ff4b4b'}, ax=ax)
    ax.set_xticklabels(['No Default','Default'])
    ax.set_title(col)

plt.suptitle('Key Features vs Default Status', fontsize=14)
plt.tight_layout()
plt.show()

## 8 · Pairplot — Top 5 Features Most Correlated with Default

In [ ]:
top5 = corr['Default'].abs().drop('Default').nlargest(5).index.tolist()

sns.pairplot(df[top5 + ['Default']], hue='Default',
             palette={0:'#00c6ff', 1:'#ff4b4b'},
             diag_kind='kde', plot_kws={'alpha': 0.4})
plt.suptitle('Pairplot — Top 5 Correlated Features', y=1.02)
plt.show()

## 9 · Summary & Key Insights

In [ ]:
print('=== KEY INSIGHTS ===')
print(f'Rows / Columns : {df.shape}')
print(f'Default rate   : {df["Default"].mean():.2%}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'\nTop 5 features correlated with Default:')
print(corr['Default'].abs().drop('Default').nlargest(5).round(3))